# Exercise 07: Unsupervised Learning – Clustering Countries with k-Means, Hierarchical Clustering and PCA

**Dataset:** socio-economic and health indicators of 167 countries, compiled for the NGO *HELP International*
(about 2017, based on World Bank data). Source:
[Kaggle – Unsupervised Learning on Country Data](https://www.kaggle.com/datasets/rohan0301/unsupervised-learning-on-country-data).

| variable | meaning |
|---|---|
| `child_mort` | deaths of children under 5 per 1'000 live births |
| `exports`, `imports` | exports / imports of goods and services per capita (% of GDP per capita) |
| `health` | total health spending per capita (% of GDP per capita) |
| `income` | net income per person (USD, purchasing power adjusted) |
| `inflation` | annual growth rate of GDP (inflation, %) |
| `life_expec` | life expectancy at birth (years) |
| `total_fer` | number of children per woman (fertility) |
| `gdpp` | GDP per capita (USD) |

**Business question:** HELP International has raised 10 million USD and wants to decide which countries need the aid
most urgently. There is **no target variable** – we look for groups of similar countries and for a compact description
of the nine indicators.

**Slides:** Part 05 «Unsupervised learning» – k-means, elbow method, hierarchical clustering (dendrogram), PCA.

**After this exercise you can ...**
- perform k-means iterations by hand and compute the within-cluster sum of squares,
- apply k-means (incl. scaling, elbow plot, several random starts) and interpret clusters,
- create and cut a dendrogram,
- use PCA to summarise many correlated features and to visualise clusters.

## Concept

**k-means.** (1) choose $k$; (2) place $k$ centres; (3) assign every point to its nearest centre; (4) move each centre to
the mean of its points; repeat (3)–(4) until nothing changes. k-means minimises the within-cluster sum of squares

$$\text{WCSS} = \sum_{c=1}^{k}\;\sum_{x_i \in C_c} \lVert x_i - \mu_c \rVert^2 \qquad(\texttt{inertia\_} \text{ in scikit-learn})$$

The result depends on the random start → use several starts (`n_init`). **Elbow method:** plot WCSS against $k$; more
clusters always reduce it – choose $k$ where the curve bends. Scale the features first and check that the clusters make
sense for the business question.

**Hierarchical (agglomerative) clustering.** Start with every observation as its own cluster; repeatedly merge the two
closest clusters until one cluster is left. The **dendrogram** shows the merge history (height = distance at which the
clusters were merged); cutting it at a height gives the clusters. *Ward linkage* merges the two clusters that increase the
WCSS the least.

**PCA (principal component analysis).** Rotates the coordinate system so that the first new axis (PC1) captures as much
of the variance as possible, PC2 the largest part of the remaining variance (orthogonal to PC1), etc. The **loadings**
show how each PC is built from the original (standardised) features; the **explained variance ratio** shows how much of
the total variance each PC captures.

## Libraries and settings

In [ ]:
# Libraries
import os
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, adjusted_rand_score
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster

# Ignore warnings
import warnings
warnings.filterwarnings('ignore')

# Show current working directory
print(os.getcwd())

## Exercise 1: k-means by hand

Six points in two dimensions: A (1, 1), B (2, 1), C (4, 3), D (5, 4), E (1, 2), F (5, 5).
We use $k = 2$ with the initial centres $\mu_1 = A = (1, 1)$ and $\mu_2 = C = (4, 3)$.

a) Assign every point to its nearest centre (Euclidean distance).

b) Compute the new centres.

c) Repeat the assignment step. Has the algorithm converged?

d) Compute the WCSS of the final solution.

e) Verify your results with `KMeans(n_clusters=2, init=np.array([[1, 1], [4, 3]]), n_init=1)`.

In [ ]:
points = pd.DataFrame({'x1': [1, 2, 4, 5, 1, 5], 'x2': [1, 1, 3, 4, 2, 5]}, index=list('ABCDEF'))

# Your code here

**Your answer:** ...

## Exercise 2: Import and explore the country data

a) Import the data, set `country` as index and show the descriptive statistics. Which countries have the highest and the
lowest child mortality?

b) Plot a correlation heatmap of the nine indicators. Which indicators are strongly correlated?

c) Why must the features be standardised before clustering?

In [ ]:
# a) Import the data
countries = pd.read_csv('./Data/country_data.csv', sep=',', index_col='country')
# Your code here

# b) Correlation heatmap
# Your code here

**Your answer:** ...

## Exercise 3: k-means – choosing k

a) Standardise the nine indicators with `StandardScaler`.

b) Fit `KMeans(n_clusters=k, n_init=10, random_state=42)` for $k = 1, \dots, 10$ and plot the WCSS (`inertia_`)
against $k$ (elbow plot). Additionally compute the silhouette score for $k = 2, \dots, 10$.

c) Which $k$ do you choose? Justify your choice also with regard to the business question.

In [ ]:
# a) Standardise
scaler = StandardScaler()
X = ___

# b) Elbow plot and silhouette score
# Your code here

**Your answer:** ...

## Exercise 4: Interpret the clusters

a) Fit k-means with $k = 3$ (`n_init=10, random_state=42`) and add the cluster labels to the data frame.

b) Compute the mean of each indicator per cluster (in the **original units**) and the number of countries per cluster.
Give each cluster a meaningful name.

c) Plot `gdpp` (logarithmic axis) against `child_mort`, coloured by cluster, and label a few countries (e.g.
Switzerland, United States, India, Nigeria, Haiti, Qatar).

d) In which cluster are Switzerland, India and Nigeria?

In [ ]:
# a) k-means with k = 3
km3 = ___
countries['cluster'] = ___

# b) Cluster profiles
# Your code here

# c) Scatter plot gdpp vs. child_mort
# Your code here

**Your answer:** ...

## Exercise 5: Stability of k-means

k-means depends on the random initial centres.

a) Fit k-means with $k = 3$ and `n_init=1` for `random_state` = 0, 1, …, 19. Print the WCSS of each run and compare the
cluster assignment with the solution of Exercise 4 using the *adjusted Rand index* (ARI: 1 = identical partition,
independent of the cluster numbers; ≈ 0 = random agreement).

b) What do you conclude? Why is `n_init=10` (the default in the slides) a good idea?

In [ ]:
# Your code here

**Your answer:** ...

## Exercise 6: Hierarchical clustering and dendrogram

a) Compute the hierarchical clustering of the standardised data with `linkage(X, method='ward')` and draw the dendrogram
with the country names as labels (`dendrogram(..., labels=countries.index.tolist(), leaf_font_size=6)`, use a wide
figure).

b) Cut the dendrogram into 3 clusters (`fcluster(Z, t=3, criterion='maxclust')`) and compare them with the k-means
clusters using `pd.crosstab`. How similar are the two solutions?

c) At which height would you cut the dendrogram? What does a long vertical line in the dendrogram mean?

In [ ]:
# a) Ward linkage and dendrogram
Z = ___
# Your code here

# b) Cut into 3 clusters and compare with k-means
# Your code here

**Your answer:** ...

## Exercise 7: PCA – summarising nine indicators

a) Fit a PCA on the standardised data. Plot the explained variance ratio of each component and the cumulative explained
variance. How many components are needed for 80 % of the variance?

b) Show the loadings of PC1 and PC2 (`pca.components_`) as a table. Interpret PC1 and PC2.

c) Plot the countries in the PC1–PC2 plane, coloured by k-means cluster, and label the countries from Exercise 4.

In [ ]:
features = ['child_mort', 'exports', 'health', 'imports', 'income',
            'inflation', 'life_expec', 'total_fer', 'gdpp']

# a) PCA and explained variance
pca = PCA()
scores = ___
# Your code here

# b) Loadings of PC1 and PC2
# Your code here

# c) Countries in the PC1-PC2 plane
# Your code here

**Your answer:** ...

## Exercise 8: Recommendation for HELP International

a) Within the cluster "in need of aid", rank the countries by their PC1 score (lowest = least developed) and show the 10
countries with the lowest scores together with `child_mort`, `life_expec`, `income` and `gdpp`.

b) Write a short recommendation (3–4 sentences) for the NGO. Which limitations of the analysis should be mentioned?

In [ ]:
# a) Countries most in need
countries['PC1'] = scores[:, 0]
# Your code here

**Your answer:** ...

### Jupyter notebook --footer info-- (please always provide this at the end of each notebook)

In [ ]:
import os
import platform
import socket
from platform import python_version
from datetime import datetime

print('-----------------------------------')
print(os.name.upper())
print(platform.system(), '|', platform.release())
print('Datetime:', datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print('Python Version:', python_version())
print('-----------------------------------')